## Creating simple agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

In [3]:
# https://openai.github.io/openai-agents-python/agents/

carbon_accounting_agent = Agent(
    name = "Carbon Accounting Expert Assistant",
    instructions = """
    You are a helpful assistant giving out carbon accounting advice.
    You give concise answer.
    """
)

### Let's execute the Agent:

In [4]:
# https://openai.github.io/openai-agents-python/tracing/
# https://platform.openai.com/logs?api=traces
with trace("Carbon Accounting Expert Assistant"):
    result = await Runner.run(carbon_accounting_agent,"How to calculate emission on bitcoin mining infrastructure?")
print(result)
print(type(result))

RunResult:
- Last agent: Agent(name="Carbon Accounting Expert Assistant", ...)
- Final output (str):
    Here’s a concise way to calculate emissions for a Bitcoin mining facility.
    
    Key inputs
    - IT load (P_IT): total electrical power used by mining rigs (kW)
    - Facility power usage efficiency (PUE): ratio of total facility power to IT power (dimensionless)
    - Hours in period (t): typically 8760 h/year
    - Emission factor (EF): CO2e per kWh for the electricity grid serving the site (kgCO2e/kWh)
    - Optional: lifecycle emissions of hardware, refrigerants, and cooling if you want a broader footprint (kgCO2e)
    
    Basic calculation (scope 2-like for electricity)
    - Total electricity consumption = P_IT × PUE × t
    - Emissions = Total electricity consumption × EF
    - If you already have total facility power (P_total), you can skip PUE: Emissions = P_total × t × EF
    
    How to get EF
    - Use regional grid EF data (e.g., national or subnational grid emissi

### Streaming the answer to the screen, token by token

In [5]:
response_stream = Runner.run_streamed(carbon_accounting_agent, "what is emission factor? explain for kids in a simple way using analogy")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

An emission factor is like a recipe card for pollution.

- It tells you how much pollution you get for doing a certain activity.
- Example: the emission factor for driving 1 kilometer tells you how much CO2 comes out per km.

So, if you know how many kilometers you drove and the emission factor (CO2 per km), you can estimate the total emissions. Units are usually kg CO2 per unit of activity (e.g., per km, per liter of fuel, per kWh of electricity).